# 1.1 电磁波偏振态三维可视化

**学习目标**
- 理解线偏振、圆偏振、椭圆偏振态的数学表示
- 掌握 Jones 矢量描述偏振态的方法
- 通过 Stokes 参数可视化偏振态的 Poincaré 球表示
- 观察电磁波传播过程中电场矢量的旋转特性

**参考文献**：Ryzhkov & Zrnic, Chapter 1, pp. 1-45

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import interact, Dropdown, FloatSlider, VBox, HTML
from IPython.display import display
import ipywidgets as widgets
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 1. 理论背景

电磁波的偏振态描述了电场矢量随时间的演化行为。对于沿 $z$ 方向传播的平面波：

$$\mathbf{E}(z,t) = \text{Re}\{[E_h(z)\hat{h} + E_v(z)\hat{v}]e^{i(kz-\omega t)}\}$$

其中 $E_h$ 和 $E_v$ 分别是水平偏振和垂直偏振分量的复振幅。

### Jones 矢量

偏振态可以用 Jones 矢量描述：

$$\mathbf{J} = \begin{pmatrix} E_h \\ E_v \end{pmatrix} = \begin{pmatrix} |E_h|e^{i\phi_h} \\ |E_v|e^{i\phi_v} \end{pmatrix}$$

归一化 Jones 矢量：$\mathbf{J}_{norm} = \frac{1}{\sqrt{|E_h|^2+|E_v|^2}}\mathbf{J}$

In [2]:
def jones_to_stokes(Eh, Ev):
    """将 Jones 矢量转换为 Stokes 参数"""
    I = np.abs(Eh)**2 + np.abs(Ev)**2
    Q = np.abs(Eh)**2 - np.abs(Ev)**2
    U = 2 * np.real(Eh * np.conj(Ev))
    V = -2 * np.imag(Eh * np.conj(Ev))
    return np.array([I, Q, U, V])

def normalize_stokes(S):
    """归一化 Stokes 参数"""
    return S / S[0] if S[0] != 0 else S

def create_polarization_widget():
    """创建偏振态选择控件"""
    polarization_type = Dropdown(
        options=['水平线偏振', '垂直线偏振', '+45°线偏振', '-45°线偏振',
                 '右旋圆偏振', '左旋圆偏振', '椭圆偏振'],
        value='水平线偏振',
        description='偏振类型:'
    )
    amplitude_h = FloatSlider(min=0.1, max=2.0, step=0.1, value=1.0, description='水平振幅:')
    amplitude_v = FloatSlider(min=0.1, max=2.0, step=0.1, value=1.0, description='垂直振幅:')
    phase_diff = FloatSlider(min=-np.pi, max=np.pi, step=0.1, value=0.0,
                            description='相位差 (rad):')
    wavelength = FloatSlider(min=0.01, max=0.1, step=0.005, value=0.03,
                            description='波长 (m):')
    return widgets.VBox([polarization_type, amplitude_h, amplitude_v, phase_diff, wavelength])

def get_jones_vector(pol_type, Ah, Av, phase_diff):
    """根据偏振类型返回 Jones 矢量"""
    if pol_type == '水平线偏振':
        return np.array([Ah, 0])
    elif pol_type == '垂直线偏振':
        return np.array([0, Av])
    elif pol_type == '+45°线偏振':
        return np.array([Ah/np.sqrt(2), Av/np.sqrt(2)])
    elif pol_type == '-45°线偏振':
        return np.array([Ah/np.sqrt(2), -Av/np.sqrt(2)])
    elif pol_type == '右旋圆偏振':
        return np.array([Ah, 1j*Av]) / np.sqrt(2)
    elif pol_type == '左旋圆偏振':
        return np.array([Ah, -1j*Av]) / np.sqrt(2)
    elif pol_type == '椭圆偏振':
        return np.array([Ah, Av * np.exp(1j * phase_diff)])
    return np.array([1.0, 0.0])

## 2. 电磁波传播可视化

下图展示电磁波在空间中传播时，电场矢量端点的轨迹。调节参数观察偏振态变化。

In [3]:
def plot_wave_propagation(pol_type='水平线偏振', Ah=1.0, Av=1.0, phase_diff=0.0, wavelength=0.03):
    """绘制电磁波传播过程中的电场矢量轨迹"""
    fig = plt.figure(figsize=(14, 5))
    
    # Jones 矢量
    jones = get_jones_vector(pol_type, Ah, Av, phase_diff)
    Eh, Ev = jones[0], jones[1]
    
    # Stokes 参数
    S = jones_to_stokes(Eh, Ev)
    S_norm = normalize_stokes(S)
    
    # ============ 子图1: 3D螺旋轨迹 ============
    ax1 = fig.add_subplot(131, projection='3d')
    
    z = np.linspace(0, 4*wavelength, 500)
    k = 2*np.pi / wavelength
    
    Ex = Ah * np.cos(k*z)  # 水平分量
    Ey = Av * np.cos(k*z + phase_diff)  # 垂直分量
    
    ax1.plot(Ex, Ey, z, 'b', linewidth=0.8)
    ax1.scatter([Ex[0]], [Ey[0]], [z[0]], color='g', s=50, label='起始点')
    ax1.set_xlabel('Ex (水平)', fontsize=10)
    ax1.set_ylabel('Ey (垂直)', fontsize=10)
    ax1.set_zlabel('z (传播方向)', fontsize=10)
    ax1.set_title('电场矢量端点轨迹', fontsize=11)
    ax1.legend()
    
    # ============ 子图2: xy平面投影 ============
    ax2 = fig.add_subplot(132)
    ax2.plot(Ex, Ey, 'b-', linewidth=0.8)
    ax2.scatter([Ex[0]], [Ey[0]], color='g', s=50, label='起始点')
    ax2.scatter([Ex[-1]], [Ey[-1]], color='r', s=50, label='终点')
    ax2.set_xlabel('Ex (水平)', fontsize=10)
    ax2.set_ylabel('Ey (垂直)', fontsize=10)
    ax2.set_title('xy平面投影（偏振椭圆）', fontsize=11)
    ax2.set_aspect('equal')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(-2, 2)
    ax2.set_ylim(-2, 2)
    
    # ============ 子图3: Stokes参数条形图 ============
    ax3 = fig.add_subplot(133)
    labels = ['I', 'Q', 'U', 'V']
    colors = ['steelblue', 'coral', 'mediumseagreen', 'orchid']
    bars = ax3.bar(labels, S_norm, color=colors, alpha=0.7, edgecolor='black')
    ax3.set_ylabel('归一化幅度', fontsize=10)
    ax3.set_title('Stokes 参数 (归一化)', fontsize=11)
    ax3.set_ylim(-1.5, 1.5)
    ax3.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    for bar, val in zip(bars, S_norm):
        ax3.text(bar.get_x() + bar.get_width()/2, val + 0.05 if val >= 0 else val - 0.1,
                f'{val:.2f}', ha='center', va='bottom' if val >= 0 else 'top', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # 显示偏振态描述
    desc = f"""偏振态信息：
    - 类型：{pol_type}
    - Stokes参数：[I={S[0]:.2f}, Q={S[1]:.2f}, U={S[2]:.2f}, V={S[3]:.2f}]
    - 归一化S：[{S_norm[0]:.2f}, {S_norm[1]:.2f}, {S_norm[2]:.2f}, {S_norm[3]:.2f}]
    - Q/I = {S_norm[1]:.2f}, U/I = {S_norm[2]:.2f}, V/I = {S_norm[3]:.2f}"""
    return desc

In [4]:
# 交互式界面
desc_html = HTML(value='')

def update_plot(pol_type, Ah, Av, phase_diff, wavelength):
    desc = plot_wave_propagation(pol_type, Ah, Av, phase_diff, wavelength)
    desc_html.value = f'<pre style="background-color:#f5f5f5;padding:10px;border-radius:5px">{desc}</pre>'

interact_widget = interact(update_plot,
    pol_type=widgets.Dropdown(options=['水平线偏振', '垂直线偏振', '+45°线偏振', '-45°线偏振',
                                      '右旋圆偏振', '左旋圆偏振', '椭圆偏振'],
                            value='水平线偏振', description='偏振类型'),
    Ah=widgets.FloatSlider(min=0.1, max=2.0, step=0.1, value=1.0, description='水平振幅'),
    Av=widgets.FloatSlider(min=0.1, max=2.0, step=0.1, value=1.0, description='垂直振幅'),
    phase_diff=widgets.FloatSlider(min=-np.pi, max=np.pi, step=0.1, value=0.0, description='相位差'),
    wavelength=widgets.FloatSlider(min=0.01, max=0.1, step=0.005, value=0.03, description='波长(m)')
)

display(desc_html)

interactive(children=(Dropdown(description='偏振类型', options=('水平线偏振', '垂直线偏振', '+45°线偏振', '-45°线偏振', '右旋圆偏振', '…

HTML(value='<pre style="background-color:#f5f5f5;padding:10px;border-radius:5px">偏振态信息：\n    - 类型：水平线偏振\n    -…

## 3. Poincaré 球可视化

Poincaré 球是表征偏振态的极坐标表示方法。球面上每一点对应一种偏振态。

点的坐标 $(S_1, S_2, S_3)$ 对应归一化 Stokes 参数 $(Q/I, U/I, V/I)$。

In [5]:
def plot_poincare_sphere(pol_type='水平线偏振', Ah=1.0, Av=1.0, phase_diff=0.0):
    """绘制 Poincaré 球和当前偏振态点"""
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # 绘制球体
    u = np.linspace(0, 2*np.pi, 50)
    v = np.linspace(0, np.pi, 50)
    X = np.outer(np.cos(u), np.sin(v))
    Y = np.outer(np.sin(u), np.sin(v))
    Z = np.outer(np.ones(np.size(u)), np.cos(v))
    
    ax.plot_surface(X, Y, Z, alpha=0.15, color='steelblue', edgecolor='none')
    
    # 绘制赤道和经线
    theta = np.linspace(0, 2*np.pi, 100)
    ax.plot(np.cos(theta), np.sin(theta), 0, 'k-', alpha=0.3, linewidth=0.5)  # 赤道
    ax.plot(0, 0, np.linspace(-1, 1, 100), 'k-', alpha=0.3, linewidth=0.5)    # 子午线
    
    # 计算当前偏振态的归一化 Stokes 参数
    jones = get_jones_vector(pol_type, Ah, Av, phase_diff)
    S = jones_to_stokes(jones[0], jones[1])
    S_norm = normalize_stokes(S)
    
    Q, U, V = S_norm[1], S_norm[2], S_norm[3]
    
    # 标记当前偏振态点
    ax.scatter([Q], [U], [V], color='red', s=150, zorder=5, label=f'{pol_type}')
    ax.plot([0, Q], [0, U], [0, V], 'r-', linewidth=2)
    
    # 标注主要偏振态位置
    ax.scatter([1], [0], [0], color='orange', s=80, marker='o')  # 水平线偏振
    ax.scatter([-1], [0], [0], color='orange', s=80, marker='o')  # 垂直线偏振
    ax.scatter([0], [1], [0], color='orange', s=80, marker='o')  # +45°线偏振
    ax.scatter([0], [-1], [0], color='orange', s=80, marker='o') # -45°线偏振
    ax.scatter([0], [0], [1], color='green', s=80, marker='^')  # 右旋圆偏振
    ax.scatter([0], [0], [-1], color='green', s=80, marker='v')  # 左旋圆偏振
    
    ax.text(1.1, 0, 0, 'H', fontsize=10)
    ax.text(-1.1, 0, 0, 'V', fontsize=10)
    ax.text(0, 1.1, 0, '+45°', fontsize=10)
    ax.text(0, -1.1, 0, '-45°', fontsize=10)
    ax.text(0, 0, 1.15, 'RCP', fontsize=10, color='green')
    ax.text(0, 0, -1.15, 'LCP', fontsize=10, color='green')
    
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_zlim(-1.3, 1.3)
    ax.set_xlabel('Q/I', fontsize=11)
    ax.set_ylabel('U/I', fontsize=11)
    ax.set_zlabel('V/I', fontsize=11)
    ax.set_title('Poincaré 球上的偏振态表示', fontsize=13)
    ax.legend(loc='upper left')
    
    plt.tight_layout()
    plt.show()
    
    return S_norm

In [6]:
# Poincaré 球交互式可视化
interact_poincare = interact(plot_poincare_sphere,
    pol_type=widgets.Dropdown(options=['水平线偏振', '垂直线偏振', '+45°线偏振', '-45°线偏振',
                                      '右旋圆偏振', '左旋圆偏振', '椭圆偏振'],
                            value='水平线偏振', description='偏振类型'),
    Ah=widgets.FloatSlider(min=0.1, max=2.0, step=0.1, value=1.0, description='水平振幅'),
    Av=widgets.FloatSlider(min=0.1, max=2.0, step=0.1, value=1.0, description='垂直振幅'),
    phase_diff=widgets.FloatSlider(min=-np.pi, max=np.pi, step=0.1, value=0.0, description='相位差')
)

interactive(children=(Dropdown(description='偏振类型', options=('水平线偏振', '垂直线偏振', '+45°线偏振', '-45°线偏振', '右旋圆偏振', '…

## 4. 偏振椭圆参数计算

对于椭圆偏振态，可以计算其椭圆率和主轴方向。

In [7]:
def calculate_ellipse_parameters(Eh, Ev):
    """计算椭圆偏振的参数"""
    a = np.abs(Eh)  # 长半轴
    b = np.abs(Ev)  # 短半轴
    
    # 椭圆率 (ellipticity)
    if a >= b:
        ellipticity = b / a if a > 0 else 0
    else:
        ellipticity = a / b if b > 0 else 0
    
    # 主轴方向角
    phase_diff = np.angle(Ev) - np.angle(Eh)
    
    return {'a': a, 'b': b, 'ellipticity': ellipticity, 'phase_diff': phase_diff}

# 示例计算
print("=== 偏振椭圆参数示例 ===")
for pol in ['水平线偏振', '右旋圆偏振', '椭圆偏振']:
    jones = get_jones_vector(pol, 1.0, 0.8, np.pi/4)
    params = calculate_ellipse_parameters(jones[0], jones[1])
    print(f"\n{pol}:")
    print(f"  长半轴 a = {params['a']:.3f}")
    print(f"  短半轴 b = {params['b']:.3f}")
    print(f"  椭圆率 = {params['ellipticity']:.3f}")

=== 偏振椭圆参数示例 ===

水平线偏振:
  长半轴 a = 1.000
  短半轴 b = 0.000
  椭圆率 = 0.000

右旋圆偏振:
  长半轴 a = 0.707
  短半轴 b = 0.566
  椭圆率 = 0.800

椭圆偏振:
  长半轴 a = 1.000
  短半轴 b = 0.800
  椭圆率 = 0.800


## 练习题

1. **改变振幅比**：将水平振幅设为1.0，垂直振幅设为0.5，观察偏振态如何从线偏振变为椭圆偏振。

2. **相位差效应**：固定振幅比 Eh/Ev = 1，将相位差从0变化到π，观察椭圆偏振态的旋向变化。

3. **圆偏振识别**：在 Poincaré 球上，右旋圆偏振和左旋圆偏振的位置有何特点？

4. ** Stokes 参数解释**：对于完全非偏振光，其 Stokes 参数 $(Q, U, V)$ 应为多少？

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 1, Springer.
- Born, M., and E. Wolf, 1999: *Principles of Optics*, 7th ed., Cambridge University Press.
- Hecht, E., 2017: *Optics*, 5th ed., Pearson.